In [10]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='groq:openai/gpt-oss-20b',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='groq:openai/gpt-oss-20b',
    tools=[square]
)

## Calling subagents

In [6]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='groq:openai/gpt-oss-20b',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [7]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='ece6da8d-5e53-477c-b067-5c7983863289'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call subagent 1 to compute sqrt(456). Use the function.', 'tool_calls': [{'id': 'fc_b3be4660-f667-4418-badc-789cd1b3f8e0', 'function': {'arguments': '{"x":456}', 'name': 'call_subagent_1'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 198, 'total_tokens': 243, 'completion_time': 0.047718417, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.011143019, 'prompt_tokens_details': None, 'queue_time': 0.016497537, 'total_time': 0.058861436}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09a90-06e2-7800-81af-bf6a9794cc55-0', tool

In [9]:
print(response["messages"][-1].content)

The square root of 456 is approximately **21.354156504**.
